# TabDPT Regressor — DIMER artifact inference tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/tabdpt-regressor-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/tabdpt-regressor-pipeline/blob/main/tutorials/tabdpt_regressor_artifact_inference_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Layer6%2FTabDPT-ffcc4d?style=flat)](https://huggingface.co/Layer6/TabDPT) [![Upstream](https://img.shields.io/badge/Upstream-layer6ai--labs%2FTabDPT--inference-181717?style=flat&logo=github&logoColor=white)](https://github.com/layer6ai-labs/TabDPT-inference) [![arXiv](https://img.shields.io/badge/arXiv-2608.01400-b31b1b.svg)](https://arxiv.org/abs/2608.01400)

**Profile:** `ARTIFACT-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification 1.1 — **standalone** (§3.6)  
**Capability:** serving-state reconstruction from an externally produced DIMER artifact (`artifact.json` + `training_context.parquet`) and point-prediction inference on genuinely new rows with the pinned `Layer6/TabDPT` v1.2 checkpoint

**This notebook is standalone.** It carries the repository's package (3 modules under `src/tabdpt_regressor_pipeline/`, at revision `d28b1854d7f1`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `4462ffbd1d8dea25d4862d30beed4b70cd596ae5` (~254 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

This notebook consumes `artifact.json` + `training_context.parquet` produced **outside this execution** (for example by the E2E tutorial in a separate session), validates the artifact and its exact pinned base-model provenance, restores the fitted preprocessing/support state, accepts genuinely new unlabelled data, predicts continuous point estimates, and exports results. No gradient fine-tuning occurs and **no artifact is created here**. The release-grade path restores the repository encoder plus the upstream fitted imputer/scaler/PCA state directly; it does not fit preprocessing on the uploaded artifact or on the inference rows. The carried package supplies `validate_artifact_bundle`, `load_verified_artifact`, `validate_inputs` and `evaluation_report`.

**Trust boundary.** Digest and manifest checks establish internal consistency, not sender authenticity. The artifact format accepts no ZIP, pickle, or arbitrary Python-object payload: the support context is Parquet, the manifest is JSON, and the base checkpoint is acquired separately (Section 3) and digest-verified before use. Use only artifacts from a trusted producer.

**Learning objectives:** install the pinned runtime, read what the carried package guarantees, resolve and digest-verify the immutable upstream checkpoint, supply an externally produced artifact and validate it before any model state is reconstructed, inspect its provenance, reconstruct the serving state through the no-refit path, validate new unlabelled rows into an input manifest, predict continuous point estimates, produce an evaluation report that is `not-measurable` because no labels exist, and export machine-readable predictions plus provenance.

**This notebook does not demonstrate:** artifact creation, in-notebook support fitting, classification, gradient fine-tuning, or any uncertainty interval or quality claim: without labelled rows nothing is measured, and the exported predictions are point estimates with no prediction interval.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.11+; the pins were executed locally on Python 3.12). GPU recommended, CPU supported but slower; FlashAttention is disabled (`use_flash=False`) for Tesla T4 portability.
- **Artifact:** an externally produced pair `artifact.json` + `training_context.parquet` (the E2E tutorial writes one to `outputs/artifact/`). Supply it through the upload dialog, or set `ARTIFACT_DIR` to a directory already present in the runtime for non-interactive execution. Nothing in this notebook manufactures it.
- **Data:** one separate, unlabelled CSV or Parquet file with exactly the artifact's fitted feature columns. It is supplied by upload or by `NEW_DATA_PATH`; no sample is bundled, because scoring self-generated rows would not be external-artifact evidence. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `Layer6/TabDPT` snapshot (~254 MB in total) at revision `4462ffbd1d8d…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `numpy`, `pandas`, `sklearn` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'tabdpt==1.2.0',
    'torch==2.7.1',
    'faiss-cpu==1.11.0',
    'huggingface-hub==0.36.2',
    'numpy==2.3.0',
    'omegaconf==2.3.0',
    'pandas==2.3.2',
    'pyarrow==25.0.0',
    'safetensors==0.5.3',
    'scikit-learn==1.7.0',
    'scipy==1.15.3',
    'tqdm==4.67.1',
]
NOTEBOOK_SOURCE = {
    'repository': 'tabdpt-regressor-pipeline',
    'repository_revision': 'd28b1854d7f1ec61d6e1070af6b4528af9db701c',
    'embedded_module': 'src/tabdpt_regressor_pipeline/pipeline.py',
    'embedded_modules': ['src/tabdpt_regressor_pipeline/pipeline.py', 'src/tabdpt_regressor_pipeline/artifact.py', 'src/tabdpt_regressor_pipeline/dimer_runtime.py'],
    'module_sha256': '8baef560e6ab7c8d05be27488aed827955ba7b3bfb00ea7cc5c666cd1a123f27',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '1.1',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, numpy, pandas, sklearn
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'numpy': numpy.__version__, 'pandas': pandas.__version__, 'sklearn': sklearn.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/tabdpt_regressor_pipeline/` @ `d28b1854d7f1`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/tabdpt_regressor_pipeline/pipeline.py`

In [ ]:
from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

TABDPT_PACKAGE_VERSION = "1.2.0"
TABDPT_UPSTREAM_CODE_COMMIT = "9cfb05e0a6bc380ae6c99c08adc8d50dacd4f246"

# Fleet snapshot identity (DIMER Notebook Specification 1.1, ST3/MOD13). The pinned upstream model is
# unchanged; these are the fleet-standard names for the same repository, revision, license and snapshot
# key. The TABDPT_* spellings below stay as the package's published names and alias these constants.
MODEL_ID = "Layer6/TabDPT"
MODEL_REVISION = "4462ffbd1d8dea25d4862d30beed4b70cd596ae5"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "tabdpt-1.2"
MANIFEST_NAME = "dimer-base-manifest.json"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative

TABDPT_HF_REPO = MODEL_ID
TABDPT_HF_REVISION = MODEL_REVISION
TABDPT_WEIGHT_FILENAME = "tabdpt1_2.safetensors"
TABDPT_WEIGHT_SHA256 = "06680220fd66c4524051706b98c1c659a674d19d3a766cd0bb276505e99faccd"

MIN_DISTINCT_TARGETS = 2  # `fit` refuses a constant target (R² would be undefined)
METRIC_IDS = ("mae", "rmse", "r2")  # the ids `evaluate` reports (target units, target units, unitless)


def sha256_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def resolve_tabdpt_weights(model_weight_path: str | Path | None = None, cache_dir: str | Path | None = None) -> Path:
    if model_weight_path is None:
        model_weight_path = hf_hub_download(
            repo_id=TABDPT_HF_REPO,
            filename=TABDPT_WEIGHT_FILENAME,
            revision=TABDPT_HF_REVISION,
            cache_dir=str(cache_dir) if cache_dir is not None else None,
        )
    path = Path(model_weight_path)
    if not path.is_file():
        raise FileNotFoundError(f"TabDPT model weight not found: {path}")
    actual = sha256_file(path)
    if actual != TABDPT_WEIGHT_SHA256:
        raise RuntimeError(
            f"TabDPT weight SHA-256 mismatch: expected {TABDPT_WEIGHT_SHA256}, got {actual}"
        )
    return path


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local pinned snapshot against its manifest; raise naming the first mismatch.

    The manifest is the parity anchor the standalone tutorial carries inline (NOTEBOOK_SPEC 1.1 ST3).
    The package's own ``TABDPT_WEIGHT_SHA256`` is not replaced by it: the manifest entry for
    ``TABDPT_WEIGHT_FILENAME`` must equal that constant, so the two can never diverge silently.
    """
    root = Path(path or DEFAULT_WEIGHTS_DIR)
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as handle:
        manifest = json.load(handle)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    entries = manifest.get("files", [])
    declared = {entry["path"]: entry["sha256"] for entry in entries}
    if declared.get(TABDPT_WEIGHT_FILENAME) != TABDPT_WEIGHT_SHA256:
        raise ValueError(
            f"manifest {TABDPT_WEIGHT_FILENAME} sha256 {declared.get(TABDPT_WEIGHT_FILENAME)!r} "
            f"!= TABDPT_WEIGHT_SHA256 {TABDPT_WEIGHT_SHA256!r}"
        )
    for entry in entries:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = sha256_file(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    hf_hub_download(
        repo_id=MODEL_ID,
        filename=relative_path,
        revision=MODEL_REVISION,
        local_dir=str(root),
    )


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a clone commits the manifest but
    git-ignores the checkpoint). Returns the relative paths fetched; ``verify_snapshot`` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as handle:
        manifest = json.load(handle)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def _resolve_use_flash(requested: bool | None, device: str | None) -> bool:
    """Enable FlashAttention by default only on CUDA devices with compute capability >= 8.0."""
    if requested is not None:
        return requested
    if device is not None and not str(device).startswith("cuda"):
        return False
    try:
        import torch

        if not torch.cuda.is_available():
            return False
        target = None if device in (None, "cuda") else device
        major, _ = torch.cuda.get_device_capability(target)
        return major >= 8
    except Exception:
        return False


class TabularFeatureEncoder:
    """Train-fitted mixed-table encoder with explicit missing/unknown categorical codes."""

    STATE_SCHEMA_VERSION = 1

    def __init__(self) -> None:
        self.feature_columns: list[str] = []
        self.numeric_columns: set[str] = set()
        self.category_maps: dict[str, dict[str, int]] = {}
        self.is_fitted = False

    def fit(self, frame: pd.DataFrame) -> TabularFeatureEncoder:
        if frame.columns.duplicated().any():
            raise ValueError("Duplicate feature column names are not supported")
        if frame.shape[1] == 0:
            raise ValueError("At least one feature column is required")
        self.feature_columns = list(frame.columns)
        self.numeric_columns = {
            col for col in self.feature_columns if pd.api.types.is_numeric_dtype(frame[col])
        }
        self.category_maps = {}
        for col in self.feature_columns:
            if col in self.numeric_columns:
                continue
            values = sorted({str(v) for v in frame[col].dropna().tolist()})
            self.category_maps[col] = {value: idx for idx, value in enumerate(values)}
        self.is_fitted = True
        return self

    def to_state(self) -> dict[str, Any]:
        if not self.is_fitted:
            raise RuntimeError("Feature encoder is not fitted")
        return {
            "schemaVersion": self.STATE_SCHEMA_VERSION,
            "featureColumns": list(self.feature_columns),
            "numericColumns": [col for col in self.feature_columns if col in self.numeric_columns],
            "categoryMaps": {
                col: dict(self.category_maps[col])
                for col in self.feature_columns
                if col in self.category_maps
            },
            "categoricalEncoding": {
                "valueNormalization": "str",
                "unknownCode": "len(categoryMap)",
                "missingCode": "len(categoryMap)+1",
            },
        }

    @classmethod
    def from_state(cls, state: dict[str, Any]) -> TabularFeatureEncoder:
        if not isinstance(state, dict) or state.get("schemaVersion") != cls.STATE_SCHEMA_VERSION:
            raise ValueError("Unsupported feature-encoder state schema")
        feature_columns = state.get("featureColumns")
        numeric_columns = state.get("numericColumns")
        category_maps = state.get("categoryMaps")
        if not isinstance(feature_columns, list) or not feature_columns or not all(isinstance(v, str) for v in feature_columns):
            raise ValueError("featureColumns must be a non-empty list of strings")
        if len(feature_columns) != len(set(feature_columns)):
            raise ValueError("featureColumns contains duplicates")
        if not isinstance(numeric_columns, list) or not all(isinstance(v, str) for v in numeric_columns):
            raise ValueError("numericColumns must be a list of strings")
        if not set(numeric_columns).issubset(feature_columns):
            raise ValueError("numericColumns must be a subset of featureColumns")
        if not isinstance(category_maps, dict):
            raise ValueError("categoryMaps must be an object")
        expected_categorical = set(feature_columns) - set(numeric_columns)
        if set(category_maps) != expected_categorical:
            raise ValueError("categoryMaps must exactly cover non-numeric feature columns")
        normalized_maps: dict[str, dict[str, int]] = {}
        for col in feature_columns:
            if col in numeric_columns:
                continue
            mapping = category_maps[col]
            if not isinstance(mapping, dict) or not all(isinstance(k, str) and isinstance(v, int) for k, v in mapping.items()):
                raise ValueError(f"Invalid category map for {col!r}")
            codes = sorted(mapping.values())
            if codes != list(range(len(codes))):
                raise ValueError(f"Category codes for {col!r} must be contiguous from zero")
            normalized_maps[col] = dict(mapping)
        encoder = cls()
        encoder.feature_columns = list(feature_columns)
        encoder.numeric_columns = set(numeric_columns)
        encoder.category_maps = normalized_maps
        encoder.is_fitted = True
        return encoder

    def transform(self, frame: pd.DataFrame) -> np.ndarray:
        if not self.is_fitted:
            raise RuntimeError("Feature encoder is not fitted")
        missing = [col for col in self.feature_columns if col not in frame.columns]
        extra = [col for col in frame.columns if col not in self.feature_columns]
        if missing or extra:
            raise ValueError(f"Feature schema mismatch; missing={missing}, extra={extra}")
        out = np.empty((len(frame), len(self.feature_columns)), dtype=np.float64)
        for idx, col in enumerate(self.feature_columns):
            series = frame[col]
            if col in self.numeric_columns:
                out[:, idx] = pd.to_numeric(series, errors="coerce").to_numpy(dtype=np.float64)
                continue
            mapping = self.category_maps[col]
            unknown_code = float(len(mapping))
            missing_code = float(len(mapping) + 1)
            encoded = []
            for value in series.tolist():
                if pd.isna(value):
                    encoded.append(missing_code)
                else:
                    encoded.append(float(mapping.get(str(value), unknown_code)))
            out[:, idx] = encoded
        return out

    def fit_transform(self, frame: pd.DataFrame) -> np.ndarray:
        return self.fit(frame).transform(frame)


def _set_deterministic_seed(seed: int | None) -> None:
    if seed is None:
        return
    import random

    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch

        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except (ImportError, AttributeError):
        pass


INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        "pandas.DataFrame, one row per example; feature columns of any dtype plus, for fit/evaluate, "
        "a numeric target column"
    ),
    "columns": "unique column names; `drop_columns` are removed before encoding",
    "target": "numeric, finite, no missing values, at least MIN_DISTINCT_TARGETS distinct values",
    "distinct_targets": [MIN_DISTINCT_TARGETS, None],
    "features": [1, None],
    "inference_input": (
        "exactly the fitted feature columns (after `drop_columns`), no target or output columns"
    ),
    "preprocessing": (
        "numeric columns are kept (NaN passes to TabDPT's support-fitted mean imputer); other columns "
        "are mapped to fitted integer codes with dedicated missing and unknown codes; upstream "
        "standardisation and any PCA basis are fitted on the support rows and reused at inference"
    ),
}


def _check_fit_inputs(
    frame: pd.DataFrame, target_column: str, drop_columns: Sequence[str] | None
) -> tuple[list[str], pd.DataFrame, pd.Series]:
    """The checks `fit` applies, in `fit`'s order, raising `fit`'s errors; returns what `fit` derives."""
    if frame.columns.duplicated().any():
        raise ValueError("Duplicate column names are not supported")
    if target_column not in frame.columns:
        raise ValueError(f"Target column {target_column!r} not found")
    drops = list(dict.fromkeys(c for c in (drop_columns or []) if c != target_column))
    raw_target = frame[target_column]
    target = pd.to_numeric(raw_target, errors="coerce")
    invalid = raw_target.notna() & target.isna()
    if invalid.any():
        examples = raw_target[invalid].astype(str).head(5).tolist()
        raise ValueError(f"Regression target contains non-numeric values: {examples}")
    if target.isna().any() or not np.isfinite(target.to_numpy(dtype=np.float64)).all():
        raise ValueError("Regression target must be finite and non-missing")
    if target.nunique() < MIN_DISTINCT_TARGETS:
        raise ValueError("Regression target must not be constant")
    features = frame.drop(columns=[target_column, *drops], errors="ignore")
    if features.shape[1] == 0:
        raise ValueError("At least one feature column is required")
    return drops, features, target


def _check_inference_inputs(
    frame: pd.DataFrame, required: Sequence[str], drop_columns: Sequence[str]
) -> pd.DataFrame:
    """The schema check `predict` applies to an inference table; returns the ordered feature frame."""
    effective = frame.drop(columns=list(drop_columns), errors="ignore")
    missing = [col for col in required if col not in effective.columns]
    extra = [col for col in effective.columns if col not in required]
    if missing or extra:
        raise ValueError(f"Feature schema mismatch; missing={missing}, extra={extra}")
    return effective.loc[:, list(required)]


def validate_inputs(
    frame: pd.DataFrame,
    target_column: str | None = "target",
    drop_columns: Sequence[str] | None = None,
    *,
    feature_columns: Sequence[str] | None = None,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observed table properties, verdict).

    With a ``target_column`` the table is checked exactly as ``fit`` checks it; with
    ``target_column=None`` it is an inference table checked against ``feature_columns`` (the fitted
    schema) exactly as ``predict`` checks it. Rejection is reported by raising the same error the
    core method raises; a caller that wants the finding recorded catches it and stores ``str(exc)``.
    """
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (the table's id)")
    table_id = names[0] if names else "table-0"
    if target_column is None:
        if feature_columns is None:
            raise ValueError("feature_columns is required to validate an inference table")
        drops = list(drop_columns or [])
        checked = _check_inference_inputs(frame, list(feature_columns), drops)
        missing_counts = checked.isna().sum()
        entry: dict[str, Any] = {
            "id": table_id,
            "mode": "inference",
            "rows": len(checked),
            "feature_columns": list(checked.columns),
            "missing_value_columns": {str(col): int(n) for col, n in missing_counts.items() if n > 0},
        }
    else:
        drops, features, target = _check_fit_inputs(frame, target_column, drop_columns)
        numeric = [col for col in features.columns if pd.api.types.is_numeric_dtype(features[col])]
        missing_counts = features.isna().sum()
        values = target.to_numpy(dtype=np.float64)
        entry = {
            "id": table_id,
            "mode": "fit",
            "rows": len(frame),
            "feature_columns": list(features.columns),
            "numeric_columns": numeric,
            "categorical_columns": [col for col in features.columns if col not in numeric],
            "missing_value_columns": {str(col): int(n) for col, n in missing_counts.items() if n > 0},
            "target_summary": {
                "min": float(values.min()),
                "max": float(values.max()),
                "mean": float(values.mean()),
                "distinct": int(target.nunique()),
            },
        }
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [entry],
        "target_column": target_column,
        "drop_columns": drops,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def training_mean_baseline(
    support_targets: Sequence[Any], holdout_targets: Sequence[Any]
) -> dict[str, float]:
    """The trivial baseline `evaluate` is compared against: always predict the support mean.

    The metric ids and their definitions are the ones ``evaluate`` reports (MAE and RMSE in target
    units, R² relative to the holdout's own mean, so the baseline's R² is at most 0).
    """
    support = pd.to_numeric(pd.Series(list(support_targets)), errors="coerce")
    holdout = pd.to_numeric(pd.Series(list(holdout_targets)), errors="coerce")
    for name, series in (("support_targets", support), ("holdout_targets", holdout)):
        if series.empty or series.isna().any() or not np.isfinite(series.to_numpy(dtype=np.float64)).all():
            raise ValueError(f"{name} must be non-empty, numeric and finite")
    if holdout.nunique() < MIN_DISTINCT_TARGETS:
        raise ValueError("Regression evaluation target must not be constant because R² is undefined")
    y_true = holdout.to_numpy(dtype=np.float64)
    constant = np.full(len(y_true), float(support.mean()))
    return {
        "mae": float(mean_absolute_error(y_true, constant)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, constant))),
        "r2": float(r2_score(y_true, constant)),
    }


def evaluation_report(
    metrics: Mapping[str, float] | None,
    *,
    baseline: Mapping[str, float] | None = None,
    n_holdout: int | None = None,
    target_column: str | None = None,
    sample_kind: str = "sample",
    estimation: str = "single seeded random holdout; no dispersion estimate",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    ``metrics`` is the dict ``evaluate`` returns (ids ``mae``, ``rmse``, ``r2``) and ``baseline`` the
    dict ``training_mean_baseline`` returns; the report is ``sample-sanity`` evidence. Without metrics
    (no labelled holdout) the verdict is ``not-measurable`` and the report says what labelled data would
    make the task measurable.
    """
    base: dict[str, Any] = {
        "task": "tabular regression by in-context conditioning on labelled support rows",
        "score_semantics": (
            "continuous point predictions in target units; no per-prediction uncertainty interval is produced"
        ),
        "sample_kind": sample_kind,
        "n_holdout": n_holdout,
        "target_column": target_column,
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if metrics is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no labelled holdout rows were supplied for the scored table",
            "needs": (
                "a labelled holdout table with a finite, non-constant numeric target column, scored with "
                "`evaluate` (mae, rmse, r2) against `training_mean_baseline`; an independent test set from "
                "the deployment domain for any generalisable claim, and calibration data before any "
                "prediction interval is attached"
            ),
        }
    unknown = sorted(set(metrics) - set(METRIC_IDS))
    if unknown:
        raise ValueError(f"unknown metric ids {unknown}; `evaluate` reports {list(METRIC_IDS)}")
    units = {"mae": "target units", "rmse": "target units", "r2": "unitless (1 - SSE/SST)"}
    reported = [
        {
            "id": metric_id,
            "value": float(metrics[metric_id]),
            "units": units[metric_id],
            "higher_is_better": metric_id == "r2",
            "estimation": estimation,
        }
        for metric_id in METRIC_IDS
        if metric_id in metrics
    ]
    baselines = []
    if baseline is not None:
        baselines.append(
            {
                "id": "training_mean",
                "metrics": [
                    {"id": metric_id, "value": float(baseline[metric_id])}
                    for metric_id in METRIC_IDS
                    if metric_id in baseline
                ],
            }
        )
    rows = "an unstated number of" if n_holdout is None else str(n_holdout)
    return {
        **base,
        "metrics": reported,
        "baselines": baselines,
        "verdict": "sample-sanity",
        "reason": f"{rows} labelled holdout row(s) from one seeded split; tutorial evidence, not a benchmark",
        "needs": (
            "an independent, domain-representative labelled test set for any generalisable quality "
            "claim; the point predictions carry no uncertainty interval"
        ),
    }


class TabDPTRegressionPipeline:
    def __init__(
        self,
        model_weight_path: str | Path | None = None,
        cache_dir: str | Path | None = None,
        device: str | None = None,
        use_flash: bool | None = None,
        compile_model: bool = False,
        verbose: bool = False,
        seed: int | None = 42,
    ) -> None:
        self.model_weight_path = model_weight_path
        self.cache_dir = cache_dir
        self.device = device
        self.use_flash = _resolve_use_flash(use_flash, device)
        self.compile_model = compile_model
        self.verbose = verbose
        self.seed = seed
        self.feature_encoder = TabularFeatureEncoder()
        self.target_column: str | None = None
        self.drop_columns_: list[str] = []
        self.estimator: Any | None = None
        self.source: str = "local-snapshot" if model_weight_path is not None else "hf-cache"

    @classmethod
    def from_pretrained(
        cls,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        **kwargs: Any,
    ) -> TabDPTRegressionPipeline:
        """Build a pipeline whose base checkpoint is the digest-verified snapshot in ``weights_dir``.

        Stages only the manifest entries that are absent (at ``MODEL_REVISION``), re-hashes every entry
        against the manifest, and then pins ``model_weight_path`` to the verified file, so the in-context
        ``fit`` that follows can load nothing else. No model is loaded here.
        """
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        stage_missing_files(root, allow_download=allow_download)
        verify_snapshot(root)
        weight_path = root / TABDPT_WEIGHT_FILENAME
        pipeline = cls(model_weight_path=weight_path, **kwargs)
        pipeline.source = "local-snapshot"
        return pipeline

    def fit(
        self,
        frame: pd.DataFrame,
        target_column: str = "target",
        drop_columns: list[str] | None = None,
        seed: int | None = None,
    ):
        # The same checks `validate_inputs` applies (one shared function, so they cannot diverge).
        drops, features, target = _check_fit_inputs(frame, target_column, drop_columns)
        if seed is not None:
            self.seed = seed
        _set_deterministic_seed(self.seed)
        self.drop_columns_ = drops
        X = self.feature_encoder.fit_transform(features)
        y = target.to_numpy(dtype=np.float64)
        weights = resolve_tabdpt_weights(self.model_weight_path, self.cache_dir)
        from tabdpt import TabDPTRegressor

        self.estimator = TabDPTRegressor(
            model_weight_path=str(weights),
            device=self.device,
            use_flash=self.use_flash,
            compile=self.compile_model,
            context_reduction="subsample",
            verbose=self.verbose,
        )
        self.estimator.fit(X, y)
        self.target_column = target_column
        return self

    def export_preprocessing_state(self) -> dict[str, Any]:
        if not self.feature_encoder.is_fitted or self.target_column is None:
            raise RuntimeError("Pipeline preprocessing state is not fitted")
        state: dict[str, Any] = {
            "schemaVersion": 1,
            "targetColumn": self.target_column,
            "dropColumns": list(self.drop_columns_),
            "seed": self.seed,
            "encoder": self.feature_encoder.to_state(),
        }
        if self.estimator is not None:
            upstream: dict[str, Any] = {"seed": self.seed}
            V = getattr(self.estimator, "V", None)
            if V is not None:
                if hasattr(V, "detach"):
                    upstream["pca_basis"] = V.detach().cpu().numpy().tolist()
                elif isinstance(V, np.ndarray):
                    upstream["pca_basis"] = V.tolist()
                elif isinstance(V, list):
                    upstream["pca_basis"] = V
                elif hasattr(V, "data"):
                    v_data = V.data
                    if hasattr(v_data, "detach"):
                        upstream["pca_basis"] = v_data.detach().cpu().numpy().tolist()
                    elif isinstance(v_data, np.ndarray):
                        upstream["pca_basis"] = v_data.tolist()
                    elif isinstance(v_data, list):
                        upstream["pca_basis"] = v_data
                    else:
                        upstream["pca_basis"] = np.asarray(v_data).tolist()
            imputer = getattr(self.estimator, "imputer", None)
            if imputer is not None and hasattr(imputer, "statistics_") and imputer.statistics_ is not None:
                upstream["imputer_statistics"] = np.asarray(imputer.statistics_).tolist()
            scaler = getattr(self.estimator, "scaler", None)
            if scaler is not None and hasattr(scaler, "mean_") and scaler.mean_ is not None:
                upstream["scaler_mean"] = np.asarray(scaler.mean_).tolist()
                upstream["scaler_scale"] = np.asarray(scaler.scale_).tolist()
            state["upstream"] = upstream
        return state

    def condition_on_context(
        self,
        context_frame: pd.DataFrame,
        preprocessing_state: dict[str, Any],
        seed: int | None = None,
    ) -> TabDPTRegressionPipeline:
        """Condition the pipeline on an in-context support table using restored preprocessing state without refitting."""
        if not isinstance(preprocessing_state, dict) or preprocessing_state.get("schemaVersion") != 1:
            raise ValueError("Unsupported preprocessing state schemaVersion")
        encoder_state = preprocessing_state.get("encoder")
        if not isinstance(encoder_state, dict):
            raise ValueError("Invalid preprocessing state: missing 'encoder'")
        target_column = preprocessing_state.get("targetColumn")
        if not target_column or target_column not in context_frame.columns:
            raise ValueError(f"Target column {target_column!r} not found in context frame")

        effective_seed = seed if seed is not None else preprocessing_state.get("seed", self.seed)
        _set_deterministic_seed(effective_seed)
        self.seed = effective_seed

        self.feature_encoder = TabularFeatureEncoder.from_state(encoder_state)
        self.target_column = target_column
        self.drop_columns_ = list(preprocessing_state.get("dropColumns", []))

        features = context_frame.drop(columns=[target_column, *self.drop_columns_], errors="ignore")
        X = self.feature_encoder.transform(features)

        raw_target = pd.to_numeric(context_frame[target_column], errors="coerce")
        if raw_target.isna().any() or not np.isfinite(raw_target.to_numpy(dtype=np.float64)).all():
            raise ValueError("Regression target in context frame must be finite and non-missing")
        if raw_target.nunique() < 2:
            raise ValueError("Regression target in context frame must not be constant")
        y = raw_target.to_numpy(dtype=np.float64)

        weights = resolve_tabdpt_weights(self.model_weight_path, self.cache_dir)
        from tabdpt import TabDPTRegressor

        self.estimator = TabDPTRegressor(
            model_weight_path=str(weights),
            device=self.device,
            use_flash=self.use_flash,
            compile=self.compile_model,
            context_reduction="subsample",
            verbose=self.verbose,
        )
        self.estimator.fit(X, y)

        upstream = preprocessing_state.get("upstream")
        if isinstance(upstream, dict):
            pca_basis = upstream.get("pca_basis")
            if pca_basis is not None and hasattr(self.estimator, "V"):
                existing_v = getattr(self.estimator, "V", None)
                target_device = (
                    getattr(existing_v, "device", None)
                    or getattr(self.estimator, "device", None)
                    or self.device
                    or "cpu"
                )
                try:
                    import torch

                    target_dtype = getattr(existing_v, "dtype", torch.float32)
                    self.estimator.V = torch.as_tensor(
                        pca_basis,
                        dtype=target_dtype,
                        device=target_device,
                    )
                except Exception:
                    if hasattr(existing_v, "device") or (target_device and str(target_device) != "cpu"):
                        from types import SimpleNamespace

                        self.estimator.V = SimpleNamespace(
                            data=np.array(pca_basis, dtype=np.float32),
                            device=target_device,
                            dtype=getattr(existing_v, "dtype", "float32"),
                        )
                    else:
                        self.estimator.V = np.array(pca_basis, dtype=np.float32)
        return self

    @classmethod
    def load_artifact(
        cls,
        artifact_path: str | Path,
        context_path: str | Path | None = None,
        model_weight_path: str | Path | None = None,
        cache_dir: str | Path | None = None,
        device: str | None = None,
        use_flash: bool | None = None,
        compile_model: bool = False,
        verbose: bool = False,
        seed: int | None = None,
    ) -> TabDPTRegressionPipeline:
        """Load a DIMER serving artifact bundle, restoring preprocessing from manifest without refitting."""
        artifact_file = Path(artifact_path)
        if not artifact_file.is_file():
            raise FileNotFoundError(f"Artifact manifest not found: {artifact_file}")
        manifest = json.loads(artifact_file.read_text(encoding="utf-8"))
        fmt = manifest.get("format")
        if fmt not in ("tabdpt-dimer-context-v3", "tabdpt-dimer-context-v2"):
            raise ValueError(f"Unsupported artifact format: {fmt!r}")
        if manifest.get("taskType") != "tabular_regression":
            raise ValueError(
                f"Artifact taskType mismatch: expected 'tabular_regression', got {manifest.get('taskType')!r}"
            )
        preprocessing = manifest.get("preprocessing")
        if not isinstance(preprocessing, dict):
            raise ValueError("Artifact manifest missing 'preprocessing' state")

        if context_path is None:
            default_name = "training_context.parquet" if fmt == "tabdpt-dimer-context-v3" else "training_context.csv"
            context_rel = manifest.get("trainingContext", {}).get("path", default_name)
            context_file = artifact_file.parent / context_rel
        else:
            context_file = Path(context_path)

        if not context_file.is_file():
            raise FileNotFoundError(f"Training context table not found: {context_file}")

        expected_sha = manifest.get("trainingContext", {}).get("sha256")
        if expected_sha:
            actual_sha = sha256_file(context_file)
            if actual_sha != expected_sha:
                raise RuntimeError(
                    f"Training context digest mismatch: expected {expected_sha}, got {actual_sha}"
                )

        encoder_state = preprocessing.get("encoder", {})
        category_cols = list(encoder_state.get("categoryMaps", {}).keys())

        suffix = context_file.suffix.lower()
        if suffix in (".parquet", ".pq"):
            try:
                context_df = pd.read_parquet(context_file, engine="pyarrow")
            except ImportError as err:
                raise ImportError(
                    "pyarrow is required to load parquet serving context in 'tabdpt-dimer-context-v3'. "
                    "Install it with 'pip install pyarrow'."
                ) from err
            for col in category_cols:
                if col in context_df.columns:
                    context_df[col] = context_df[col].astype("string")
        else:
            dtype_spec = {col: "string" for col in category_cols}
            context_df = pd.read_csv(context_file, dtype=dtype_spec)
        effective_seed = seed if seed is not None else preprocessing.get("seed", 42)
        pipeline = cls(
            model_weight_path=model_weight_path,
            cache_dir=cache_dir,
            device=device,
            use_flash=use_flash,
            compile_model=compile_model,
            verbose=verbose,
            seed=effective_seed,
        )
        pipeline.condition_on_context(context_df, preprocessing, seed=effective_seed)
        return pipeline

    def _require_fitted(self):
        if self.estimator is None or self.target_column is None:
            raise RuntimeError("Pipeline is not fitted")

    def _feature_frame(self, frame: pd.DataFrame) -> pd.DataFrame:
        return _check_inference_inputs(frame, self.feature_encoder.feature_columns, self.drop_columns_)

    def predict(
        self,
        frame: pd.DataFrame,
        n_ensembles: int = 4,
        context_size: int | None = 2048,
        batch_size: int | None = 4096,
        seed: int = 42,
    ) -> pd.Series:
        self._require_fitted()
        X = self.feature_encoder.transform(self._feature_frame(frame))
        pred = self.estimator.predict(
            X,
            n_ensembles=n_ensembles,
            context_size=context_size,
            batch_size=batch_size,
            seed=seed,
        )
        return pd.Series(np.asarray(pred, dtype=np.float64), index=frame.index, name="prediction")

    def evaluate(self, frame: pd.DataFrame, **kwargs) -> dict[str, float]:
        self._require_fitted()
        if self.target_column not in frame.columns:
            raise ValueError(f"Evaluation target {self.target_column!r} not found")
        y_true = pd.to_numeric(frame[self.target_column], errors="coerce")
        if y_true.isna().any() or not np.isfinite(y_true.to_numpy(dtype=np.float64)).all():
            raise ValueError("Evaluation target must be finite and numeric")
        if y_true.nunique() < 2:
            raise ValueError("Regression evaluation target must not be constant because R² is undefined")
        features = frame.drop(columns=[self.target_column])
        pred = self.predict(features, **kwargs)
        mse = mean_squared_error(y_true, pred)
        return {
            "mae": float(mean_absolute_error(y_true, pred)),
            "rmse": float(np.sqrt(mse)),
            "r2": float(r2_score(y_true, pred)),
        }

**Module 2/3:** `src/tabdpt_regressor_pipeline/artifact.py` (carried verbatim; see the note above)

In [ ]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path, PurePosixPath
from types import SimpleNamespace
from typing import Any

import numpy as np
import pandas as pd

# standalone rewrite (build_notebook.py): `from .pipeline import (` removed — names are kernel globals defined by the carried modules

ARTIFACT_FORMAT = "tabdpt-dimer-context-v3"
ARTIFACT_FORMAT_VERSION = 3
ARTIFACT_SEMANTICS = "support-context-plus-pinned-base-model"

EXPECTED_BASE_MODEL = {
    "repo": TABDPT_HF_REPO,
    "revision": TABDPT_HF_REVISION,
    "filename": TABDPT_WEIGHT_FILENAME,
    "sha256": TABDPT_WEIGHT_SHA256,
    "upstreamCodeCommit": TABDPT_UPSTREAM_CODE_COMMIT,
}


class _RestoredMeanImputer:
    """Minimal inference-only mean imputer reconstructed from fitted statistics."""

    def __init__(self, statistics: np.ndarray) -> None:
        self.statistics_ = np.asarray(statistics, dtype=np.float64)

    def transform(self, values: np.ndarray) -> np.ndarray:
        array = np.asarray(values, dtype=np.float64)
        if array.ndim != 2 or array.shape[1] != len(self.statistics_):
            raise ValueError("Restored imputer input width does not match fitted statistics")
        if np.isinf(array).any():
            raise ValueError("Inference features must not contain infinite values")
        return np.where(np.isnan(array), self.statistics_[None, :], array)


class _RestoredStandardScaler:
    """Minimal inference-only StandardScaler equivalent reconstructed from fitted state."""

    def __init__(self, mean: np.ndarray, scale: np.ndarray) -> None:
        self.mean_ = np.asarray(mean, dtype=np.float64)
        self.scale_ = np.asarray(scale, dtype=np.float64)

    def transform(self, values: np.ndarray) -> np.ndarray:
        array = np.asarray(values, dtype=np.float64)
        if array.ndim != 2 or array.shape[1] != len(self.mean_):
            raise ValueError("Restored scaler input width does not match fitted statistics")
        return (array - self.mean_[None, :]) / self.scale_[None, :]


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _safe_bundle_member(manifest_path: Path, relative_path: str) -> Path:
    if not relative_path or "\\" in relative_path:
        raise ValueError("Artifact member path must be a non-empty POSIX relative path")
    posix_path = PurePosixPath(relative_path)
    if posix_path.is_absolute() or ".." in posix_path.parts:
        raise ValueError(f"Unsafe artifact member path: {relative_path!r}")
    root = manifest_path.parent.resolve()
    resolved = (root / Path(*posix_path.parts)).resolve()
    if resolved != root and root not in resolved.parents:
        raise ValueError(f"Artifact member escapes bundle root: {relative_path!r}")
    return resolved


def _finite_vector(value: Any, name: str, width: int, *, positive: bool = False) -> np.ndarray:
    try:
        array = np.asarray(value, dtype=np.float64)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"Artifact {name} must be a numeric vector") from exc
    if array.shape != (width,):
        raise ValueError(f"Artifact {name} must contain exactly {width} values")
    if not np.isfinite(array).all():
        raise ValueError(f"Artifact {name} must contain only finite values")
    if positive and not np.all(array > 0):
        raise ValueError(f"Artifact {name} must contain only positive values")
    return array


def _restore_state(
    preprocessing: dict[str, Any],
    width: int,
    *,
    require_complete: bool,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray | None] | None:
    upstream = preprocessing.get("upstream")
    required = ("imputer_statistics", "scaler_mean", "scaler_scale")
    if not isinstance(upstream, dict) or any(key not in upstream for key in required):
        if require_complete:
            raise ValueError(
                "Artifact is missing fitted upstream preprocessing state required for no-refit reconstruction"
            )
        return None

    imputer_statistics = _finite_vector(
        upstream["imputer_statistics"], "preprocessing.upstream.imputer_statistics", width
    )
    scaler_mean = _finite_vector(
        upstream["scaler_mean"], "preprocessing.upstream.scaler_mean", width
    )
    scaler_scale = _finite_vector(
        upstream["scaler_scale"], "preprocessing.upstream.scaler_scale", width, positive=True
    )

    pca_basis = upstream.get("pca_basis")
    if pca_basis is not None:
        try:
            pca_basis = np.asarray(pca_basis, dtype=np.float32)
        except (TypeError, ValueError) as exc:
            raise ValueError("Artifact preprocessing.upstream.pca_basis must be numeric") from exc
        if pca_basis.ndim != 2 or pca_basis.shape[0] != width or not np.isfinite(pca_basis).all():
            raise ValueError(
                "Artifact preprocessing.upstream.pca_basis must be a finite 2D matrix with one row per encoded feature"
            )
    return imputer_statistics, scaler_mean, scaler_scale, pca_basis


def export_artifact_bundle(
    pipeline: TabDPTRegressionPipeline,
    training_context: pd.DataFrame,
    output_dir: str | Path,
) -> Path:
    """Export the reusable DIMER serving state for an in-context TabDPT regressor."""
    if pipeline.target_column is None or not pipeline.feature_encoder.is_fitted:
        raise RuntimeError("Pipeline must be conditioned before artifact export")
    if training_context.columns.duplicated().any():
        raise ValueError("Training context contains duplicate column names")
    if pipeline.target_column not in training_context.columns:
        raise ValueError(f"Training context is missing target column {pipeline.target_column!r}")

    expected_features = list(pipeline.feature_encoder.feature_columns)
    feature_frame = training_context.drop(
        columns=[pipeline.target_column, *pipeline.drop_columns_], errors="ignore"
    )
    if list(feature_frame.columns) != expected_features:
        raise ValueError(
            "Training context schema does not match the fitted serving schema; "
            f"expected={expected_features}, got={list(feature_frame.columns)}"
        )
    pipeline.feature_encoder.transform(feature_frame)

    preprocessing = pipeline.export_preprocessing_state()
    _restore_state(preprocessing, len(expected_features), require_complete=True)

    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    context_path = output / "training_context.parquet"
    training_context.to_parquet(context_path, index=False)

    manifest = {
        "format": ARTIFACT_FORMAT,
        "formatVersion": ARTIFACT_FORMAT_VERSION,
        "taskType": "tabular_regression",
        "artifactSemantics": ARTIFACT_SEMANTICS,
        "targetColumn": pipeline.target_column,
        "dropColumns": list(pipeline.drop_columns_),
        "preprocessing": preprocessing,
        "baseModel": dict(EXPECTED_BASE_MODEL),
        "trainingContext": {
            "path": context_path.name,
            "size": context_path.stat().st_size,
            "sha256": _sha256(context_path),
        },
    }
    manifest_path = output / "artifact.json"
    manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return manifest_path


def validate_artifact_bundle(artifact_path: str | Path) -> tuple[dict[str, Any], Path]:
    """Validate artifact identity and context integrity before model-state reconstruction.

    Existing DIMER v3 manifests encode their version in ``format`` and may omit the newer
    explicit ``formatVersion``, ``artifactSemantics``, and context ``size`` fields. When those
    additive fields are present they are validated. New explicit-version artifacts must also
    carry complete fitted upstream preprocessing state so verified serving can reconstruct it
    without fitting preprocessing again.
    """
    manifest_path = Path(artifact_path)
    if not manifest_path.is_file() or manifest_path.is_symlink():
        raise ValueError(f"Artifact manifest must be a regular file: {manifest_path}")
    try:
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    except json.JSONDecodeError as exc:
        raise ValueError("Artifact manifest is not valid JSON") from exc
    if not isinstance(manifest, dict):
        raise ValueError("Artifact manifest must contain a JSON object")
    if manifest.get("format") != ARTIFACT_FORMAT:
        raise ValueError(f"Unsupported artifact format: {manifest.get('format')!r}")

    explicit_version = manifest.get("formatVersion")
    if explicit_version is not None and explicit_version != ARTIFACT_FORMAT_VERSION:
        raise ValueError(
            f"Unsupported artifact formatVersion: {explicit_version!r}; expected {ARTIFACT_FORMAT_VERSION}"
        )
    explicit_semantics = manifest.get("artifactSemantics")
    if explicit_semantics is not None and explicit_semantics != ARTIFACT_SEMANTICS:
        raise ValueError(f"Unsupported artifact semantics: {explicit_semantics!r}")
    if manifest.get("taskType") != "tabular_regression":
        raise ValueError(f"Artifact taskType mismatch: {manifest.get('taskType')!r}")

    base_model = manifest.get("baseModel")
    if not isinstance(base_model, dict):
        raise ValueError("Artifact manifest is missing baseModel provenance")
    for key, expected in EXPECTED_BASE_MODEL.items():
        actual = base_model.get(key)
        if actual != expected:
            raise ValueError(
                f"Artifact baseModel.{key} mismatch: expected {expected!r}, got {actual!r}"
            )

    preprocessing = manifest.get("preprocessing")
    if not isinstance(preprocessing, dict):
        raise ValueError("Artifact manifest is missing preprocessing state")
    if preprocessing.get("targetColumn") != manifest.get("targetColumn"):
        raise ValueError("Artifact targetColumn disagrees with preprocessing state")
    if list(preprocessing.get("dropColumns", [])) != list(manifest.get("dropColumns", [])):
        raise ValueError("Artifact dropColumns disagree with preprocessing state")
    encoder_state = preprocessing.get("encoder")
    if not isinstance(encoder_state, dict):
        raise ValueError("Artifact preprocessing state is missing encoder state")
    feature_columns = encoder_state.get("featureColumns")
    if not isinstance(feature_columns, list) or not feature_columns:
        raise ValueError("Artifact encoder featureColumns must be a non-empty list")
    _restore_state(
        preprocessing,
        len(feature_columns),
        require_complete=explicit_version is not None or explicit_semantics is not None,
    )

    context = manifest.get("trainingContext")
    if not isinstance(context, dict):
        raise ValueError("Artifact manifest is missing trainingContext")
    context_path = _safe_bundle_member(manifest_path, context.get("path", ""))
    if not context_path.is_file() or context_path.is_symlink():
        raise ValueError(f"Training context must be a regular file: {context_path}")

    expected_size = context.get("size")
    if expected_size is not None:
        if not isinstance(expected_size, int) or expected_size < 0:
            raise ValueError("Training context size is invalid")
        if expected_size != context_path.stat().st_size:
            raise ValueError(
                f"Training context size mismatch: expected {expected_size}, got {context_path.stat().st_size}"
            )
    expected_sha = context.get("sha256")
    if not isinstance(expected_sha, str) or len(expected_sha) != 64:
        raise ValueError("Training context SHA-256 is missing or malformed")
    actual_sha = _sha256(context_path)
    if actual_sha != expected_sha:
        raise ValueError(
            f"Training context SHA-256 mismatch: expected {expected_sha}, got {actual_sha}"
        )
    return manifest, context_path


def _read_context(context_path: Path, preprocessing: dict[str, Any]) -> pd.DataFrame:
    encoder_state = preprocessing["encoder"]
    category_columns = list(encoder_state.get("categoryMaps", {}).keys())
    suffix = context_path.suffix.lower()
    if suffix not in (".parquet", ".pq"):
        raise ValueError("Verified v3 artifacts require a Parquet training context")
    try:
        context = pd.read_parquet(context_path, engine="pyarrow")
    except ImportError as exc:
        raise ImportError("pyarrow is required to load verified v3 artifact context") from exc
    for column in category_columns:
        if column in context.columns:
            context[column] = context[column].astype("string")
    return context


def _pca_on_estimator_device(estimator: Any, pca_basis: np.ndarray) -> Any:
    target_device = getattr(estimator, "device", None) or "cpu"
    try:
        import torch

        return torch.as_tensor(pca_basis, dtype=torch.float32, device=target_device)
    except Exception:
        return SimpleNamespace(data=np.asarray(pca_basis, dtype=np.float32), device=target_device, dtype="float32")


def _reconstruct_without_preprocessing_refit(
    manifest: dict[str, Any],
    context: pd.DataFrame,
    *,
    model_weight_path: str | Path | None,
    cache_dir: str | Path | None,
    device: str | None,
    use_flash: bool,
    compile_model: bool,
    verbose: bool,
    seed: int | None,
) -> TabDPTRegressionPipeline | None:
    preprocessing = manifest["preprocessing"]
    encoder_state = preprocessing["encoder"]
    effective_seed = seed if seed is not None else preprocessing.get("seed", 42)
    _set_deterministic_seed(effective_seed)

    pipeline = TabDPTRegressionPipeline(
        model_weight_path=model_weight_path,
        cache_dir=cache_dir,
        device=device,
        use_flash=use_flash,
        compile_model=compile_model,
        verbose=verbose,
        seed=effective_seed,
    )
    pipeline.feature_encoder = TabularFeatureEncoder.from_state(encoder_state)
    pipeline.target_column = preprocessing.get("targetColumn")
    pipeline.drop_columns_ = list(preprocessing.get("dropColumns", []))

    if not pipeline.target_column or pipeline.target_column not in context.columns:
        raise ValueError(f"Target column {pipeline.target_column!r} not found in artifact context")
    features = context.drop(columns=[pipeline.target_column, *pipeline.drop_columns_], errors="ignore")
    X = pipeline.feature_encoder.transform(features)
    y_series = pd.to_numeric(context[pipeline.target_column], errors="coerce")
    if y_series.isna().any() or not np.isfinite(y_series.to_numpy(dtype=np.float64)).all():
        raise ValueError("Regression target in artifact context must be finite and numeric")
    if y_series.nunique() < 2:
        raise ValueError("Regression target in artifact context must not be constant")
    y = y_series.to_numpy(dtype=np.float64)

    restore = _restore_state(
        preprocessing,
        X.shape[1],
        require_complete=manifest.get("formatVersion") is not None or manifest.get("artifactSemantics") is not None,
    )
    if restore is None:
        return None
    imputer_statistics, scaler_mean, scaler_scale, pca_basis = restore

    weights = resolve_tabdpt_weights(model_weight_path, cache_dir)
    from tabdpt import TabDPTRegressor

    estimator = TabDPTRegressor(
        model_weight_path=str(weights),
        device=device,
        use_flash=use_flash,
        compile=compile_model,
        context_reduction="subsample",
        verbose=verbose,
    )
    if getattr(estimator, "missing_indicators", False):
        raise ValueError("Artifact reconstruction supports TabDPT missing_indicators=False only")
    normalizer = getattr(estimator, "normalizer", "standard")
    if normalizer != "standard":
        raise ValueError(f"Artifact reconstruction expected TabDPT standard normalizer, got {normalizer!r}")

    imputer = _RestoredMeanImputer(imputer_statistics)
    scaler = _RestoredStandardScaler(scaler_mean, scaler_scale)
    X_train = scaler.transform(imputer.transform(X))

    estimator.imputer = imputer
    estimator.scaler = scaler
    estimator.X_train = X_train
    estimator.y_train = y
    estimator.n_instances, estimator.n_features = X_train.shape
    estimator.faiss_knn = None
    estimator.is_fitted_ = True

    # Test doubles and diagnostics may expose raw conditioning arrays under X/y.
    if hasattr(estimator, "X"):
        estimator.X = X
    if hasattr(estimator, "y"):
        estimator.y = y

    feature_reduction = getattr(estimator, "feature_reduction", "pca")
    max_features = int(getattr(estimator, "max_features", X_train.shape[1]))
    reduction_required = X_train.shape[1] > max_features
    if feature_reduction == "pca" and reduction_required:
        if pca_basis is None:
            raise ValueError("Wide artifact requires the fitted PCA basis for no-refit reconstruction")
        estimator.V = _pca_on_estimator_device(estimator, pca_basis)
    elif pca_basis is not None:
        estimator.V = _pca_on_estimator_device(estimator, pca_basis)
    else:
        estimator.V = None

    if compile_model and hasattr(estimator, "model") and hasattr(estimator.model, "compile"):
        estimator.model.compile()

    pipeline.estimator = estimator
    pipeline.preprocessing_restored_ = True
    return pipeline


def load_verified_artifact(
    artifact_path: str | Path,
    *,
    model_weight_path: str | Path | None = None,
    cache_dir: str | Path | None = None,
    device: str | None = None,
    use_flash: bool = False,
    compile_model: bool = False,
    verbose: bool = False,
    seed: int | None = None,
) -> TabDPTRegressionPipeline:
    """Validate an artifact and reconstruct serving state without re-fitting saved preprocessing.

    New explicit-version artifacts must contain fitted upstream imputer/scaler state and use the
    no-refit reconstruction path. Older v3 artifacts without that complete additive state remain
    loadable through the legacy compatibility path, which reconditions from the saved support
    table and is therefore not used as release-grade artifact-inference evidence.
    """
    manifest_path = Path(artifact_path)
    manifest, context_path = validate_artifact_bundle(manifest_path)
    context = _read_context(context_path, manifest["preprocessing"])
    restored = _reconstruct_without_preprocessing_refit(
        manifest,
        context,
        model_weight_path=model_weight_path,
        cache_dir=cache_dir,
        device=device,
        use_flash=use_flash,
        compile_model=compile_model,
        verbose=verbose,
        seed=seed,
    )
    if restored is not None:
        return restored

    pipeline = TabDPTRegressionPipeline.load_artifact(
        manifest_path,
        context_path=context_path,
        model_weight_path=model_weight_path,
        cache_dir=cache_dir,
        device=device,
        use_flash=use_flash,
        compile_model=compile_model,
        verbose=verbose,
        seed=seed,
    )
    pipeline.preprocessing_restored_ = False
    return pipeline

**Module 3/3:** `src/tabdpt_regressor_pipeline/dimer_runtime.py` (carried verbatim; see the note above)

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import zipfile
from dataclasses import asdict, dataclass
from pathlib import Path, PurePosixPath
from typing import Any

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# standalone rewrite (build_notebook.py): `from .pipeline import (` removed — names are kernel globals defined by the carried modules

SUPPORTED_PREPROCESSING_KEYS = frozenset(
    {"target_column", "drop_columns", "max_train_rows", "validation_split"}
)
SUPPORTED_HYPERPARAMETER_KEYS = frozenset(
    {"fine_tune", "n_ensembles", "context_size", "batch_size", "seed"}
)
TRANSPORT_HYPERPARAMETER_KEYS = frozenset({"model_id"})


def _json_object(value: str | None, name: str) -> dict[str, Any]:
    if not value:
        return {}
    try:
        parsed = json.loads(value)
    except json.JSONDecodeError as exc:
        raise ValueError(f"{name} must contain valid JSON") from exc
    if not isinstance(parsed, dict):
        raise ValueError(f"{name} must contain a JSON object")
    return parsed


def _reject_unknown(
    payload: dict[str, Any],
    supported: frozenset[str],
    name: str,
    transport_keys: frozenset[str] = frozenset(),
) -> None:
    unknown = sorted(set(payload) - supported - transport_keys)
    if unknown:
        raise ValueError(f"Unsupported {name} keys: {unknown}")


def _integer(payload: dict[str, Any], key: str, default: int, minimum: int, maximum: int) -> int:
    value = payload.get(key, default)
    if isinstance(value, bool) or not isinstance(value, int):
        raise ValueError(f"{key} must be an integer")
    if not minimum <= value <= maximum:
        raise ValueError(f"{key} must be between {minimum} and {maximum}")
    return value


def _number(payload: dict[str, Any], key: str, default: float, minimum: float, maximum: float) -> float:
    value = payload.get(key, default)
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise ValueError(f"{key} must be numeric")
    value = float(value)
    if not minimum <= value <= maximum:
        raise ValueError(f"{key} must be between {minimum} and {maximum}")
    return value


def _drop_columns(value: Any) -> list[str]:
    if value in (None, ""):
        return []
    if isinstance(value, str):
        items = [item.strip() for item in value.split(",")]
    elif isinstance(value, list) and all(isinstance(item, str) for item in value):
        items = [item.strip() for item in value]
    else:
        raise ValueError("drop_columns must be a comma-separated string or list of strings")
    return list(dict.fromkeys(item for item in items if item))


@dataclass(frozen=True)
class DimerRuntimeConfig:
    target_column: str = "target"
    drop_columns: tuple[str, ...] = ()
    max_train_rows: int = 10000
    validation_split: float = 0.2
    fine_tune: bool = False
    n_ensembles: int = 4
    context_size: int = 2048
    batch_size: int = 4096
    seed: int = 42

    @classmethod
    def from_payloads(
        cls, preprocessing: dict[str, Any] | None = None, hyperparameters: dict[str, Any] | None = None
    ) -> DimerRuntimeConfig:
        pre = dict(preprocessing or {})
        hp = dict(hyperparameters or {})
        _reject_unknown(pre, SUPPORTED_PREPROCESSING_KEYS, "preprocessing")
        _reject_unknown(
            hp,
            SUPPORTED_HYPERPARAMETER_KEYS,
            "hyperparameter",
            transport_keys=TRANSPORT_HYPERPARAMETER_KEYS,
        )
        target_column = pre.get("target_column", "target")
        if not isinstance(target_column, str) or not target_column.strip() or len(target_column) > 128:
            raise ValueError("target_column must be a non-empty string of at most 128 characters")
        fine_tune = hp.get("fine_tune", False)
        if not isinstance(fine_tune, bool):
            raise ValueError("fine_tune must be boolean")
        if fine_tune:
            raise ValueError("TabDPT v1.2 does not support gradient fine-tuning; fine_tune must be false")
        return cls(
            target_column=target_column.strip(),
            drop_columns=tuple(_drop_columns(pre.get("drop_columns", ""))),
            max_train_rows=_integer(pre, "max_train_rows", 10000, 200, 50000),
            validation_split=_number(pre, "validation_split", 0.2, 0.05, 0.4),
            fine_tune=False,
            n_ensembles=_integer(hp, "n_ensembles", 4, 1, 16),
            context_size=_integer(hp, "context_size", 2048, 128, 16384),
            batch_size=_integer(hp, "batch_size", 4096, 1, 131072),
            seed=_integer(hp, "seed", 42, 0, 2147483647),
        )

    @classmethod
    def from_environment(cls) -> DimerRuntimeConfig:
        return cls.from_payloads(
            _json_object(os.getenv("DIMER_PREPROCESSING_ARGS_JSON"), "DIMER_PREPROCESSING_ARGS_JSON"),
            _json_object(os.getenv("DIMER_HYPERPARAMETERS_JSON"), "DIMER_HYPERPARAMETERS_JSON"),
        )

    def inference_kwargs(self) -> dict[str, Any]:
        return {
            "n_ensembles": self.n_ensembles,
            "context_size": self.context_size,
            "batch_size": self.batch_size,
            "seed": self.seed,
        }


def _dataset_limits() -> tuple[int, int, int, float, int]:
    try:
        limits = (
            int(os.getenv("DIMER_MAX_ARCHIVE_BYTES", str(1024**3))),
            int(os.getenv("DIMER_MAX_UNCOMPRESSED_BYTES", str(2 * 1024**3))),
            int(os.getenv("DIMER_MAX_MEMBER_BYTES", str(512 * 1024**2))),
            float(os.getenv("DIMER_MAX_COMPRESSION_RATIO", "200")),
            int(os.getenv("DIMER_MAX_DATASET_FILES", "200")),
        )
    except ValueError as exc:
        raise ValueError("DIMER dataset safety limits must be numeric") from exc
    if not math.isfinite(limits[3]):
        raise ValueError("DIMER_MAX_COMPRESSION_RATIO must be finite")
    if any(value <= 0 for value in limits):
        raise ValueError("DIMER dataset safety limits must be positive")
    return limits


def _normalize_member(name: str) -> str | None:
    if not name or name.endswith("/"):
        return None
    normalized = name.replace("\\", "/")
    while normalized.startswith("./"):
        normalized = normalized[2:]
    path = PurePosixPath(normalized)
    if path.is_absolute() or ".." in path.parts or (path.parts and path.parts[0].endswith(":")):
        raise ValueError(f"Unsafe dataset archive path: {name!r}")
    return path.as_posix() or None


def _validate_archive(path: Path) -> list[str]:
    max_archive, max_uncompressed, max_member, max_ratio, max_files = _dataset_limits()
    if path.is_symlink():
        raise ValueError("Dataset ZIP must not be a symlink")
    if path.stat().st_size > max_archive:
        raise ValueError("Dataset ZIP exceeds DIMER_MAX_ARCHIVE_BYTES")
    members: list[str] = []
    seen: set[str] = set()
    total = 0
    try:
        with zipfile.ZipFile(path) as archive:
            for info in archive.infolist():
                normalized = _normalize_member(info.filename)
                if normalized is None:
                    continue
                if info.file_size > max_member:
                    raise ValueError(f"Archive member {normalized!r} exceeds DIMER_MAX_MEMBER_BYTES")
                total += info.file_size
                if total > max_uncompressed:
                    raise ValueError("Dataset ZIP exceeds DIMER_MAX_UNCOMPRESSED_BYTES")
                ratio = info.file_size / max(info.compress_size, 1)
                if ratio > max_ratio:
                    raise ValueError(
                        f"Archive member {normalized!r} exceeds DIMER_MAX_COMPRESSION_RATIO"
                    )
                if normalized in seen:
                    raise ValueError(f"Duplicate normalized archive path: {normalized!r}")
                seen.add(normalized)
                members.append(info.filename)
    except zipfile.BadZipFile as exc:
        raise ValueError(f"Invalid ZIP archive: {exc}") from exc
    if len(members) > max_files:
        raise ValueError(
            f"Dataset ZIP contains {len(members)} files; DIMER_MAX_DATASET_FILES={max_files}"
        )
    return members


def _validate_direct_dataset(root: Path) -> None:
    _, max_uncompressed, max_member, _, max_files = _dataset_limits()
    files: list[Path] = []
    for path in sorted(root.rglob("*")):
        if path.is_symlink():
            raise ValueError(f"Dataset directory must not contain symlinks: {path.relative_to(root)}")
        if path.is_file():
            files.append(path)
    if len(files) > max_files:
        raise ValueError(
            f"Dataset directory contains {len(files)} files; DIMER_MAX_DATASET_FILES={max_files}"
        )
    total = 0
    for path in files:
        size = path.stat().st_size
        if size > max_member:
            raise ValueError(f"Dataset file {path.name!r} exceeds DIMER_MAX_MEMBER_BYTES")
        total += size
        if total > max_uncompressed:
            raise ValueError("Dataset directory exceeds DIMER_MAX_UNCOMPRESSED_BYTES")


def _find_csv(root: Path, stem: str, required: bool) -> Path | None:
    matches = sorted(path for path in root.rglob("*.csv") if path.stem.lower() == stem.lower())
    if len(matches) > 1:
        raise ValueError(f"Multiple {stem}.csv files found")
    if not matches:
        if required:
            raise ValueError(f"Dataset must contain {stem}.csv")
        return None
    return matches[0]


def _zip_member(members: list[str], stem: str, required: bool) -> str | None:
    matches = sorted(
        name
        for name in members
        if Path(name).suffix.lower() == ".csv" and Path(name).stem.lower() == stem.lower()
    )
    if len(matches) > 1:
        raise ValueError(f"Archive contains multiple {stem}.csv files")
    if not matches:
        if required:
            raise ValueError(f"Archive must contain {stem}.csv")
        return None
    return matches[0]


def load_dimer_tables(dataset_dir: str | Path) -> tuple[pd.DataFrame, pd.DataFrame | None]:
    root = Path(dataset_dir)
    if root.is_symlink():
        raise ValueError("DIMER_DATASET_DIR must not be a symlink")
    if not root.is_dir():
        raise ValueError(f"DIMER_DATASET_DIR is not a directory: {root}")
    archives = sorted(path for path in root.iterdir() if path.is_file() and path.suffix.lower() == ".zip")
    direct_csv = list(root.rglob("*.csv"))
    if archives:
        if len(archives) != 1 or direct_csv:
            raise ValueError("Dataset directory must contain either CSV files or exactly one ZIP archive")
        members = _validate_archive(archives[0])
        with zipfile.ZipFile(archives[0]) as archive:
            train_name = _zip_member(members, "train", required=True)
            val_name = _zip_member(members, "val", required=False)
            if train_name is None:
                raise ValueError("Archive must contain train.csv")
            with archive.open(train_name) as handle:
                train = pd.read_csv(handle)
            val = None
            if val_name is not None:
                with archive.open(val_name) as handle:
                    val = pd.read_csv(handle)
            return train, val
    _validate_direct_dataset(root)
    train_path = _find_csv(root, "train", required=True)
    val_path = _find_csv(root, "val", required=False)
    if train_path is None:
        raise ValueError("Dataset must contain train.csv")
    return pd.read_csv(train_path), pd.read_csv(val_path) if val_path is not None else None


def _validate_target(frame: pd.DataFrame, config: DimerRuntimeConfig, name: str) -> None:
    if config.target_column not in frame.columns:
        raise ValueError(f"{name}.csv is missing target column {config.target_column!r}")
    raw = frame[config.target_column]
    target = pd.to_numeric(raw, errors="coerce")
    invalid = raw.notna() & target.isna()
    if invalid.any() or target.isna().any() or not np.isfinite(target.to_numpy(dtype=np.float64)).all():
        raise ValueError(f"{name}.csv target must be finite and numeric")
    if target.nunique() < 2:
        if name == "train":
            raise ValueError("Regression training target must not be constant")
        raise ValueError("Regression validation target must not be constant")


def prepare_dimer_frames(
    train: pd.DataFrame, val: pd.DataFrame | None, config: DimerRuntimeConfig
) -> tuple[pd.DataFrame, pd.DataFrame]:
    _validate_target(train, config, "train")
    if val is None:
        train, val = train_test_split(
            train,
            test_size=config.validation_split,
            random_state=config.seed,
        )
    else:
        _validate_target(val, config, "val")
    if len(train) > config.max_train_rows:
        train = train.sample(n=config.max_train_rows, random_state=config.seed)
    if val is None:
        raise RuntimeError("Validation split was not created")
    _validate_target(train, config, "train")
    _validate_target(val, config, "val")
    return train.reset_index(drop=True), val.reset_index(drop=True)


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    temp.replace(path)


def run_dimer_job() -> dict[str, Any]:
    config = DimerRuntimeConfig.from_environment()
    dataset_dir = os.getenv("DIMER_DATASET_DIR", "/data/dataset")
    output_dir = Path(os.getenv("DIMER_OUTPUT_DIR", "/data/output"))
    result_path = Path(os.getenv("DIMER_RESULT_PATH", str(output_dir / "result.json")))
    train, val = load_dimer_tables(dataset_dir)
    train, val = prepare_dimer_frames(train, val, config)

    weight_path = os.getenv("DIMER_BASE_MODEL_PATH", "").strip() or None
    pipeline = TabDPTRegressionPipeline(
        model_weight_path=weight_path,
        device=os.getenv("DIMER_DEVICE", "").strip() or None,
        compile_model=False,
        verbose=False,
    )
    pipeline.fit(train, target_column=config.target_column, drop_columns=list(config.drop_columns))
    metrics = pipeline.evaluate(val, **config.inference_kwargs())

    artifact_dir = output_dir / "artifacts"
    artifact_dir.mkdir(parents=True, exist_ok=True)
    context_path = artifact_dir / "training_context.parquet"
    train.to_parquet(context_path, index=False)
    manifest_path = artifact_dir / "artifact.json"
    preprocessing_state = pipeline.export_preprocessing_state()
    manifest = {
        "format": "tabdpt-dimer-context-v3",
        "taskType": "tabular_regression",
        "targetColumn": config.target_column,
        "dropColumns": list(preprocessing_state["dropColumns"]),
        "runtimeConfig": asdict(config),
        "preprocessing": preprocessing_state,
        "baseModel": {
            "repo": TABDPT_HF_REPO,
            "revision": TABDPT_HF_REVISION,
            "filename": TABDPT_WEIGHT_FILENAME,
            "sha256": TABDPT_WEIGHT_SHA256,
            "upstreamCodeCommit": TABDPT_UPSTREAM_CODE_COMMIT,
        },
        "trainingContext": {"path": context_path.name, "sha256": _sha256(context_path)},
    }
    _write_json(manifest_path, manifest)
    result = {
        "contractVersion": 1,
        "successful": True,
        "metadata": {
            "taskType": "tabular_regression",
            "runId": os.getenv("DIMER_RUN_ID", ""),
            "trainRows": len(train),
            "validationRows": len(val),
            "artifactFormat": manifest["format"],
        },
        "metrics": metrics,
        "artifacts": {
            "trainingContext": {"path": str(context_path), "sha256": _sha256(context_path)},
            "manifest": {"path": str(manifest_path), "sha256": _sha256(manifest_path)},
        },
        "provenance": {"baseModelSha256": TABDPT_WEIGHT_SHA256, "baseModelRevision": TABDPT_HF_REVISION},
    }
    _write_json(result_path, result)
    return result


def main() -> int:
    try:
        run_dimer_job()
        return 0
    except Exception as exc:
        output_dir = Path(os.getenv("DIMER_OUTPUT_DIR", "/data/output"))
        result_path = Path(os.getenv("DIMER_RESULT_PATH", str(output_dir / "result.json")))
        _write_json(
            result_path,
            {
                "contractVersion": 1,
                "successful": False,
                "metadata": {"taskType": "tabular_regression", "runId": os.getenv("DIMER_RUN_ID", "")},
                "error": {"type": type(exc).__name__, "message": str(exc)},
            },
        )
        return 1


if False:  # standalone rewrite (build_notebook.py): `if __name__ == "__main__":` disabled — the cell runs as __main__
    raise SystemExit(main())

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `1`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `4462ffbd1d8d…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `TabDPTRegressionPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, compile_model=False, use_flash=False, seed=42)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "tabdpt-1.2",
  "modelId": "Layer6/TabDPT",
  "revision": "4462ffbd1d8dea25d4862d30beed4b70cd596ae5",
  "files": [
    {
      "path": "tabdpt1_2.safetensors",
      "bytes": 254098072,
      "sha256": "06680220fd66c4524051706b98c1c659a674d19d3a766cd0bb276505e99faccd"
    }
  ],
  "totalBytes": 254098072
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = TabDPTRegressionPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, compile_model=False, use_flash=False, seed=42)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Supply the external artifact and validate it before any model state is reconstructed

Leave `ARTIFACT_DIR` empty to upload exactly `artifact.json` and `training_context.parquet`; set it to a directory already in the runtime to skip the dialog (an executor places the files there). `validate_artifact_bundle` runs **before** reconstruction and checks format/task, the support table's path/size/SHA-256, fitted-preprocessing consistency, and the manifest's base-model repository/revision/filename/digest/upstream commit against the carried package contract — a mismatch stops the notebook. Release-grade artifacts carry explicit format metadata and complete fitted upstream preprocessing state; older v3 artifacts stay loadable through a legacy compatibility path, and Section 5 fails closed when the loader reports that path was used. The cell prints the provenance a consumer needs: format, version, base model, target column, feature schema and context digest.

In [ ]:
import json
import os

ARTIFACT_DIR = ''  # @param {type:"string"}
os.makedirs('outputs', exist_ok=True)
if ARTIFACT_DIR:
    ART = Path(ARTIFACT_DIR)
    artifact_source = f'directory: {ART}'
else:
    from google.colab import files
    ART = Path('external-artifact')
    ART.mkdir(parents=True, exist_ok=True)
    uploaded = files.upload()
    required = {'artifact.json', 'training_context.parquet'}
    if set(uploaded) != required:
        raise ValueError(f'Upload exactly {sorted(required)}; got {sorted(uploaded)}')
    for name, payload in uploaded.items():
        (ART / name).write_bytes(payload)
    artifact_source = 'upload dialog'
manifest_path = ART / 'artifact.json'
manifest, context_path = validate_artifact_bundle(manifest_path)
encoder_state = manifest['preprocessing']['encoder']
expected = list(encoder_state['featureColumns'])
numeric_columns = list(encoder_state['numericColumns'])
categorical_columns = list(encoder_state['categoryMaps'].keys())
target_column = manifest['preprocessing']['targetColumn']
print({'artifact_source': artifact_source, 'format': manifest['format'], 'formatVersion': manifest.get('formatVersion', 'legacy-v3-implicit'), 'artifactSemantics': manifest.get('artifactSemantics')})
print(json.dumps(manifest['baseModel'], indent=2))
print({'targetColumn': target_column, 'featureCount': len(expected), 'numericColumns': numeric_columns, 'categoricalColumns': categorical_columns, 'contextBytes': context_path.stat().st_size, 'contextSha256': manifest['trainingContext']['sha256']})

## 5. Reconstruct the serving state with fitted preprocessing restored

`load_verified_artifact` re-validates the artifact, restores the serialised feature encoder, the fitted upstream mean-imputation and standardisation state and any saved PCA basis, and re-registers the support table as TabDPT context. The base checkpoint is the digest-verified file from Section 3 (`pipe.model_weight_path`), so **no network fallback** can substitute another model. This is serving-state reconstruction for in-context inference, not gradient training, and the preprocessing statistics are not fitted again — the cell fails closed if the loader reports the legacy reconditioning path or a target-column disagreement. It also surfaces the model feature ceiling and whether feature reduction is active; `context_size` limits the support rows used at prediction time.

In [ ]:
serving = load_verified_artifact(manifest_path, model_weight_path=pipe.model_weight_path, compile_model=False, use_flash=False)
if getattr(serving, 'preprocessing_restored_', False) is not True:
    raise RuntimeError('This artifact used the legacy compatibility/reconditioning path. Supply a release artifact with complete fitted preprocessing state.')
if serving.target_column != target_column:
    raise RuntimeError('Reconstructed target column disagrees with the validated manifest.')
encoded_feature_count = len(serving.feature_encoder.feature_columns)
model_feature_ceiling = int(serving.estimator.max_features)
feature_reduction = str(serving.estimator.feature_reduction)
print({'target': serving.target_column, 'encodedFeatureCount': encoded_feature_count, 'modelFeatureCeiling': model_feature_ceiling, 'featureReduction': feature_reduction, 'featureReductionActive': encoded_feature_count > model_feature_ceiling, 'preprocessingRestoredWithoutRefit': True, 'baseWeight': str(pipe.model_weight_path)})

## 6. Supply new unlabelled rows → validate → input manifest

Leave `NEW_DATA_PATH` empty to upload one CSV or Parquet file, or set it to a file already in the runtime. The file must contain exactly the fitted feature columns printed above, with no target or pre-existing `prediction` column. For CSV, categorical columns are read explicitly as strings so values such as `01` are not silently converted to numbers; Parquet categorical columns are likewise cast. Non-numeric or infinite values in numeric columns fail clearly, before any imputation.

`validate_inputs(..., target_column=None, feature_columns=...)` is the pipeline's public validation stage for inference tables: it applies exactly the schema check `predict` applies and returns an **input manifest** naming the schema, the row count and the missing-value columns; it is written to `outputs/tabdpt_regressor_artifact_inference_input_manifest.json`. To show what rejection looks like, the cell also validates a probe with one fitted column removed and records the pipeline's own error message as a finding. Missing categorical values use the fitted missing code, unseen categories the fitted unknown code, and numeric NaN is transformed by the **restored** training-fitted mean imputer; nothing is fitted on inference data.

In [ ]:
import csv
import io

NEW_DATA_PATH = ''  # @param {type:"string"}
if NEW_DATA_PATH:
    input_name, raw = os.path.basename(NEW_DATA_PATH), Path(NEW_DATA_PATH).read_bytes()
else:
    new_upload = files.upload()
    if len(new_upload) != 1:
        raise ValueError('Upload exactly one CSV or Parquet input.')
    input_name, raw = next(iter(new_upload.items()))
if input_name.lower().endswith('.csv'):
    header = next(csv.reader(io.StringIO(raw.decode('utf-8-sig'))), [])
    duplicates = sorted({x for x in header if header.count(x) > 1})
    if duplicates:
        raise ValueError(f'Duplicate CSV columns: {duplicates}')
    new_data = pd.read_csv(io.BytesIO(raw), dtype={c: 'string' for c in categorical_columns if c in header})
elif input_name.lower().endswith(('.parquet', '.pq')):
    new_data = pd.read_parquet(io.BytesIO(raw), engine='pyarrow')
    for column in categorical_columns:
        if column in new_data.columns:
            new_data[column] = new_data[column].astype('string')
else:
    raise ValueError('Input must be CSV or Parquet.')
if new_data.columns.duplicated().any():
    raise ValueError('Duplicate columns are not supported.')
reserved = [serving.target_column, 'prediction']
present_reserved = [c for c in reserved if c in new_data]
if present_reserved:
    raise ValueError(f'Remove target/prediction columns before inference: {present_reserved}')
print({'ceilings': {'MIN_DISTINCT_TARGETS': MIN_DISTINCT_TARGETS, 'modelFeatureCeiling': model_feature_ceiling}})
input_manifest = validate_inputs(new_data, None, feature_columns=expected, names=[input_name])
# Demonstrate rejection on a probe that breaks the fitted schema; the finding is recorded, not swallowed.
try:
    validate_inputs(new_data.drop(columns=[expected[0]]), None, feature_columns=expected)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'missing-column-probe', 'verdict': 'rejected', 'message': str(exc)})
new_data = new_data.loc[:, expected].copy()
for column in numeric_columns:
    original = new_data[column]
    converted = pd.to_numeric(original, errors='coerce')
    if (original.notna() & converted.isna()).any():
        raise ValueError(f'Numeric feature {column!r} contains non-numeric values.')
    finite = converted.dropna().to_numpy(dtype=float)
    if finite.size and not np.isfinite(finite).all():
        raise ValueError(f'Numeric feature {column!r} contains infinite values.')
    new_data[column] = converted
with open('outputs/tabdpt_regressor_artifact_inference_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 7. Predict, report what cannot be measured, and export

`predict()` returns **continuous point estimates only** in the target's units — no prediction interval is produced, so any tolerance band is the caller's to set on labelled data. `row_id` maps each prediction to its input row. `evaluation_report` is the pipeline's public evaluation stage and is produced even here: with no labelled rows its verdict is `not-measurable` and it states what labelled data would make the task measurable; it is written to `outputs/tabdpt_regressor_artifact_inference_evaluation_report.json`. The prediction CSV carries `row_id` and `prediction`, and the result JSON records the externally supplied artifact identity, the immutable model contract, the target column, the restored preprocessing/capacity state, the inference configuration, the input shape, the notebook's source and the runtime identity; it contains no credentials.

In [ ]:
kw = {'n_ensembles': 2, 'context_size': 512, 'batch_size': 512, 'seed': 42}
context_rows = len(pd.read_parquet(context_path, engine='pyarrow'))
print('Requested context_size:', kw['context_size'], 'effective support rows <=', min(context_rows, kw['context_size']))
pred = serving.predict(new_data, **kw)
results = pd.DataFrame({'row_id': new_data.index.to_numpy(), 'prediction': pred.to_numpy()})
results.to_csv('outputs/tabdpt_regressor_artifact_inference_predictions.csv', index=False)
report = evaluation_report(None, n_holdout=0, target_column=serving.target_column, sample_kind='BYOD')
with open('outputs/tabdpt_regressor_artifact_inference_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
payload = {
    'predictions': results.to_dict(orient='records'),
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'artifact': {'source': artifact_source, 'format': manifest['format'], 'formatVersion': manifest.get('formatVersion'), 'baseModel': manifest['baseModel'], 'targetColumn': serving.target_column, 'trainingContextSha256': manifest['trainingContext']['sha256'], 'contextRows': context_rows},
    'preprocessing': {'preprocessingRestoredWithoutRefit': True, 'encodedFeatureCount': encoded_feature_count, 'modelFeatureCeiling': model_feature_ceiling, 'featureReduction': feature_reduction, 'featureReductionActive': encoded_feature_count > model_feature_ceiling, 'numericMissingPolicy': 'restored training-fitted mean imputation', 'categoricalMissingPolicy': 'restored fitted missing code', 'unknownCategoryPolicy': 'restored fitted unknown code'},
    'inference': {**kw, 'output': 'continuous point predictions in target units', 'uncertaintyInterval': None, 'effectiveSupportRowsAtMost': min(context_rows, kw['context_size'])},
    'input': {'filename': input_name, 'rows': len(new_data), 'features': list(new_data.columns)},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'upstream_code_commit': TABDPT_UPSTREAM_CODE_COMMIT,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'tabdpt': importlib.metadata.version('tabdpt'), 'numpy': numpy.__version__, 'pandas': pandas.__version__, 'sklearn': sklearn.__version__, 'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU', 'use_flash': False},
}
with open('outputs/tabdpt_regressor_artifact_inference_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(results.head())
print(json.dumps(report, indent=2))
print(sorted(os.listdir('outputs')))

## Interpretation and limits

A successful run proves that an independently supplied release artifact is internally consistent with the carried package's pinned model contract, that the fitted preprocessing was restored without refitting, that the support context was reconstructed, and that schema-compatible new records were scored with explicit capacity/context behaviour as continuous point estimates — without the repository being reachable. It does **not** authenticate the producer or establish predictive quality, robustness, calibration, fairness, or production fitness; the evaluation report says `not-measurable` because no labels exist here, and the exported point estimates carry no uncertainty interval. Never bypass a failed manifest, digest, target-column, preprocessing-state or schema check; obtain a correct trusted artifact. If base-model acquisition fails, the only acceptable checkpoint is the exact `tabdpt1_2.safetensors` named by the inline manifest, never a substitute.

Successful execution proves that the recorded repository revision's package, carried in this notebook, can acquire and digest-verify the pinned checkpoint, validate and reconstruct an external artifact, validate the supplied inference table, execute the public prediction path and emit the shown machine-readable outputs in the tested runtime. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** score rows with a deliberately unseen categorical value and inspect the fitted unknown-code policy in the manifest; compare predictions at `n_ensembles` 1 versus 4; hand a labelled copy of the same rows to `evaluate` in the E2E tutorial to obtain a `sample-sanity` report with MAE / RMSE / R².

## References

- Repository README: https://github.com/kurtvalcorza/tabdpt-regressor-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/tabdpt-regressor-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/tabdpt-regressor-pipeline/blob/main/weights/README.md
- E2E companion (produces the artifact): https://github.com/kurtvalcorza/tabdpt-regressor-pipeline/blob/main/tutorials/tabdpt_regressor_colab.ipynb
- Upstream model: https://huggingface.co/Layer6/TabDPT
- Upstream inference code: https://github.com/layer6ai-labs/TabDPT-inference
- TabDPT paper: https://arxiv.org/abs/2608.01400